In [ ]:
import unicodedata
import numpy as np
import pandas as pd
from keras.src.utils.module_utils import tensorflow
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ctypes

In [ ]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)

In [ ]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=60000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

In [ ]:
train=data["train"]
train[:10]

In [ ]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [ ]:
len(data)

In [ ]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [ ]:
en_sen="Im very happy."
preprossing(en_sen)

In [ ]:
fa_sen="درود بر تو."
preprossing(fa_sen)

In [ ]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [ ]:
vocab_size=25000
max_length=35
batch_size=64

token_en=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")
token_fa=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]




In [ ]:
latent_dim=128

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True,return_sequences=True,dropout=0.4)(encoder_embedding)




decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm",dropout=0.4)
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])

attention_layers=MultiHeadAttention(num_heads=4,key_dim=32,name='multi_head_attention')

attn=attention_layers(query=decoder_outputs,key=encoder_output,value=encoder_output)
x=decoder_outputs+attn
nor=LayerNormalization(name="layer_norm")
x=nor(x)

In [ ]:
decoder_dense=Dense(vocab_size,activation="softmax",kernel_regularizer=tf.keras.regularizers.l2(1e-4))
decoder_outputs=decoder_dense(x)

In [ ]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [ ]:
model.summary()

In [ ]:
splt=int(len(en_seq)*0.9)
train_ds=tf.data.Dataset.from_tensor_slices(((en_seq[:splt],decoder_inputs_array[:splt]),decoder_targets_array[:splt])).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_seq[splt:],decoder_inputs_array[splt:]),decoder_targets_array[splt:])).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),metrics=["accuracy"],loss=tf.keras.losses.SparseCategoricalCrossentropy())

In [ ]:
history=model.fit(train_ds,epochs=100,validation_data=val_ds,verbose=2)

In [ ]:
reverse_fa = {v: k for k, v in token_fa.word_index.items()}

In [ ]:
import pickle
model.save('Translator.keras')

In [ ]:
from tensorflow.keras.models import load_model
import pickle
model=load_model("Translator.keras")
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [ ]:
import pickle

token_fa=pickle.load(open("token_fa.pkl","rb"))
token_en=pickle.load(open("token_en.pkl","rb"))
reverse_fa=pickle.load(open("reverse_fa.pkl","rb"))


In [ ]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [ ]:
decoder_state_input_h=Input(shape=(latent_dim,))
decoder_state_input_c=Input(shape=(latent_dim,))
enc_out_input=Input(shape=(max_length,latent_dim))
decoder_states_inputs=[decoder_state_input_h, decoder_state_input_c]

decoder_emb2=decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

attn2=attention_layers(query=decoder_outputs2,key=enc_out_input,value=enc_out_input)
x2=decoder_outputs2+attn2
x2=nor(x2)
decoder_outputs2 = decoder_dense(x2)

decoder_model=tf.keras.Model(
    [decoder_inputs,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)



In [ ]:
encoder_model.save('encoder_model.keras')
decoder_model.save('decoder_model.keras')

In [ ]:
def translate(sentence):

    sentence=preprossing(sentence)

    seq=token_en.texts_to_sequences([sentence])
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out,h,c=encoder_model.predict(seq)
    target_seq=np.array([[token_fa.word_index["<start>"]]])

    stop=False
    decoded= ""

    while not stop:

        output_tokens, h, c=decoder_model.predict([target_seq,h,c,enc_out])
        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word== "<end>" or len(decoded.split())>max_length:
            stop=True
        else:
            decoded+= " " +sampled_word

        target_seq=np.array([[sampled_token_index]])
        states=[h, c]

    return decoded


In [ ]:
print(translate("I love you"))


In [ ]:
print(translate('you can'))

In [ ]:
import sacrebleu

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(token_fa.index_word[1])
print(token_fa.index_word[2])
print(token_fa.index_word[3])
print(token_fa.index_word[4])
print(token_fa.index_word[5])

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
en_lengths = [len(s.split()) for s in df["en"]]
fa_lengths = [len(s.split()) for s in df["fa"]]

print("EN max:", max(en_lengths))
print("EN avg:", sum(en_lengths)//len(en_lengths))
print("FA max:", max(fa_lengths))
print("FA avg:", sum(fa_lengths)//len(fa_lengths))